In [ ]:
import json

import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

In [ ]:
def read_jsonl(path: str) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = [json.loads(l) for l in f.readlines()]

    df = pd.DataFrame(data)

    return df

In [ ]:
bf16_logs_path = "../fp8/bf16/training_logs.jsonl"
fp8_logs_path = "../fp8/fp8/training_logs.jsonl"

fp8 = read_jsonl(Path(bf16_logs_path).resolve())
bf16 = read_jsonl(Path(fp8_logs_path).resolve())

In [ ]:
SMOOTH_WINDOW = 50

train_metrics = [
    "loss",
    "learning_rate",
    "grad_norm",
    "mean_token_accuracy",
    "time_per_steps",
    "train_tokens_per_second",
    "peak_mem_alloc_gib",
    "peak_mem_reserved_gib",
]

for metric in train_metrics:
    plt.figure()
    plt.title(f"{metric} FP8 vs BF16")
    plt.xlabel("Step")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    for (dtype, df) in [("fp8", fp8), ("bf16", bf16)]:
        train_df = df[df.name == "training_loss"].copy()
        steps = train_df["data"].apply(lambda x: x.get("step"))

        y = pd.to_numeric(train_df[metric])
        if SMOOTH_WINDOW > 1:
            y_plot = y.rolling(SMOOTH_WINDOW, min_periods=1).mean()
        else:
            y_plot = y
    
        plt.plot(steps, y_plot, label=dtype, alpha=0.4)

    plt.legend()
    plt.show()

eval_metrics = [
    "eval_loss",
    "eval_mean_token_accuracy"
]

for metric in eval_metrics:
    plt.figure()
    plt.title(f"{metric} FP8 vs BF16")
    plt.xlabel("Step")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

    for (dtype, df) in [("fp8", fp8), ("bf16", bf16)]:
        eval_df = df[df.name == "validation_loss"].copy()
        steps = eval_df["data"].apply(lambda x: x.get("step"))

        y = pd.to_numeric(eval_df[metric])
        if SMOOTH_WINDOW > 1:
            y_plot = y.rolling(SMOOTH_WINDOW, min_periods=1).mean()
        else:
            y_plot = y

        plt.scatter(steps, y_plot, label=dtype, alpha=0.4)

    plt.legend(loc='lower left')
    plt.show()